In [ ]:
!pip install torch torchvision pytorch-lightning

In [ ]:
!pip install --upgrade torchmetrics

In [ ]:
# Imports

import os
import torch
import cv2
import kagglehub
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint
import albumentations as A


In [ ]:
# File paths

downloaded_data_dir = kagglehub.dataset_download("andrewmvd/car-plate-detection")
print(f"Downloaded to {downloaded_data_dir}")

images_dir = os.path.join(downloaded_data_dir, "images")
annotations_dir = os.path.join(downloaded_data_dir, "annotations")


In [ ]:
# Augmentations

train_transform = A.Compose([
    A.HueSaturationValue(10, 15, 10, p=0.3),
], bbox_params=A.BboxParams(
    format='pascal_voc',
    label_fields=['labels']
))

val_transform = A.Compose([], bbox_params=A.BboxParams(
    format='pascal_voc',
    label_fields=['labels']
))

In [ ]:
# Dataset loading

class LicensePlateDataset(Dataset):
    def __init__(self, images_dir, annotations_dir, transforms=None):
        self.images_dir = images_dir
        self.annotations_dir = annotations_dir
        self.transforms = transforms
        self.image_files = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        xml_name = img_name.replace(".png", ".xml").replace(".jpg", ".xml")
        xml_path = os.path.join(self.annotations_dir, xml_name)

        if not os.path.exists(xml_path):
            return self.__getitem__((idx + 1) % len(self))

        tree = ET.parse(xml_path)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):
            xmin = int(obj.find("bndbox/xmin").text)
            ymin = int(obj.find("bndbox/ymin").text)
            xmax = int(obj.find("bndbox/xmax").text)
            ymax = int(obj.find("bndbox/ymax").text)

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(1)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)

        if self.transforms:
            transformed = self.transforms(
                image=img,
                bboxes=boxes,
                labels=labels
            )
            img = transformed["image"]
            boxes = transformed["bboxes"]

        img = torch.tensor(img).permute(2, 0, 1).float() / 255.0
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }


        return img, target

In [ ]:
# Divide dataset into training and validation datasets


full_dataset = LicensePlateDataset(images_dir, annotations_dir)

train_size = int(0.75 * len(full_dataset))
val_size = len(full_dataset) - train_size

indices = torch.randperm(len(full_dataset)).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(
    LicensePlateDataset(images_dir, annotations_dir, transforms=train_transform),
    train_indices
)

val_dataset = Subset(
    LicensePlateDataset(images_dir, annotations_dir, transforms=val_transform),
    val_indices
)

In [ ]:
# Intersection over union calculation

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    return interArea / (boxAArea + boxBArea - interArea + 1e-6)

In [ ]:
# Create the 2 data loaders for the training and validation datasets

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

In [ ]:
# Create the actual training loop

class FasterRCNNModule(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.model = fasterrcnn_resnet50_fpn(pretrained=True)

        num_classes = 2
        in_features = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

        self.map_metric = MeanAveragePrecision(iou_type="bbox")

        self.val_outputs = []

    def forward(self, images, targets=None):
        return self.model(images, targets)

    def training_step(self, batch, batch_idx):
        images, targets = batch
        images = [img.to(self.device) for img in images]
        targets = [{k: v.to(self.device) for k, v in t.items()} for t in targets]

        loss_dict = self.model(images, targets)
        loss = sum(loss_dict.values())

        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, targets = batch
        images = [img.to(self.device) for img in images]

        outputs = self.model(images)

        preds = []
        gts = []

        for i in range(len(outputs)):
            preds.append({
                "boxes": outputs[i]["boxes"].detach().cpu(),
                "scores": outputs[i]["scores"].detach().cpu(),
                "labels": outputs[i]["labels"].detach().cpu(),
            })

            gts.append({
                "boxes": targets[i]["boxes"].detach().cpu(),
                "labels": targets[i]["labels"].detach().cpu(),
            })

        self.map_metric.update(preds, gts)

        self.val_outputs.append((outputs, targets))

    def on_validation_epoch_end(self):
        metrics = self.map_metric.compute()

        map50 = metrics["map_50"]
        map_all = metrics["map"]

        TP, FP, FN = 0, 0, 0

        for outputs, targets in self.val_outputs:
            for i in range(len(outputs)):
                pred_boxes = outputs[i]["boxes"].detach().cpu().numpy()
                scores = outputs[i]["scores"].detach().cpu().numpy()
                gt_boxes = targets[i]["boxes"].detach().cpu().numpy()

                keep = scores >= 0.5
                pred_boxes = pred_boxes[keep]

                matched = set()

                for pred_box in pred_boxes:
                    found = False
                    for j, gt_box in enumerate(gt_boxes):
                        iou = compute_iou(pred_box, gt_box)
                        if iou >= 0.5 and j not in matched:
                            TP += 1
                            matched.add(j)
                            found = True
                            break
                    if not found:
                        FP += 1

                FN += len(gt_boxes) - len(matched)

        precision = TP / (TP + FP + 1e-6)
        recall = TP / (TP + FN + 1e-6)

        self.log("mAP_50", map50, prog_bar=True)
        self.log("mAP", map_all, prog_bar=False)
        self.log("precision", precision, prog_bar=True)
        self.log("recall", recall, prog_bar=True)

        self.map_metric.reset()
        self.val_outputs.clear()

    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=0.005, momentum=0.9)

In [ ]:
# Use tensorboard for looking at the metrics

%load_ext tensorboard
%tensorboard --logdir /content/logs/faster_rcnn/

In [ ]:
# Start training

logger = TensorBoardLogger("logs", name="faster_rcnn")

model = FasterRCNNModule()

checkpoint_callback = ModelCheckpoint(
    monitor="mAP",
    mode="max",
    save_top_k=1,
    filename="best-model"
)

trainer = pl.Trainer(
    max_epochs=80,
    accelerator="gpu",
    devices=1,
    precision="16-mixed",
    logger=logger,
    callbacks=[checkpoint_callback],
)

trainer.fit(model, train_loader, val_loader)

In [ ]:
# To continue training

trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    ckpt_path="/content/logs/faster_rcnn/version_0/checkpoints/best-model.ckpt",
)

In [ ]:
# To copy the weights to google drive

# from google.colab import drive
# drive.mount('/content/drive')

# import shutil

# source_path = "/content/logs/faster_rcnn/version_2/checkpoints/best-model.ckpt"
# destination_path = "/content/drive/MyDrive/machine-learning/cmps-261/MEM/Project/FasterRCNN/best-model.ckpt"

# # Make sure destination folder exists
# import os
# os.makedirs(os.path.dirname(destination_path), exist_ok=True)

# # Copy file
# shutil.copy(source_path, destination_path)

# print("File copied successfully!")

## OCR Extension

In [ ]:
# Install

!pip install easyocr

In [ ]:
# Imports

import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import cv2
import matplotlib.pyplot as plt
import easyocr

In [ ]:
# Recreate the Faster-RCNN model

ckpt_path = "/content/drive/MyDrive/machine-learning/cmps-261/MEM/Project/FasterRCNN/best-model.ckpt"

def get_model():
    model = fasterrcnn_resnet50_fpn(pretrained=False)

    num_classes = 2
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

model = get_model()

ckpt = torch.load(ckpt_path, map_location="cpu")

model.load_state_dict({k.replace("model.", ""): v for k, v in ckpt["state_dict"].items()})

model.eval()
model.cuda()

print("Detection model loaded")

In [ ]:
# Create EasyOCR model

reader = easyocr.Reader(['en'], gpu=True)

print("OCR model ready")

In [ ]:
# Create the pipeline

def run_full_pipeline(image_path, score_threshold=0.5):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    img_tensor = torch.tensor(image_rgb).permute(2, 0, 1).float() / 255.0
    img_tensor = img_tensor.cuda()

    with torch.no_grad():
        outputs = model([img_tensor])[0]

    boxes = outputs["boxes"].cpu().numpy()
    scores = outputs["scores"].cpu().numpy()

    results = []

    for box, score in zip(boxes, scores):
        if score < score_threshold:
            continue

        x1, y1, x2, y2 = map(int, box)

        plate_crop = image_rgb[y1:y2, x1:x2]

        ocr_result = reader.readtext(plate_crop)

        text = " ".join([res[1] for res in ocr_result])

        results.append({
            "box": (x1, y1, x2, y2),
            "text": text
        })

        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(image_rgb, text, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    plt.figure(figsize=(10,6))
    plt.imshow(image_rgb)
    plt.axis("off")
    plt.show()

    return results

In [ ]:
# Final Test

from google.colab import files
import os

uploaded = files.upload()

image_path = list(uploaded.keys())[0]

results = run_full_pipeline(image_path)

for r in results:
    print(r)